# 01_validate_schema — Schema Validation

Inspect the raw collected data (`data/raw/`).

**Purpose of this notebook**
1. Load the For Sale / Sold snapshots.
2. Enumerate all fields actually present in the response.
3. Check per-field missing rate / coverage.
4. Confirm how many listings have a value for the **target (`annual_listing_multiple`)**.
5. Confirm that leakage-prone fields (price fields such as `listing_price`) actually exist.

> This step is for looking at and understanding the data.

In [ ]:
import json
from pathlib import Path
import pandas as pd

# This notebook is assumed to live in notebooks/local/.
# The project root is two levels up (../../).
PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd().name == "local") else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw folder:", RAW_DIR)
print("Raw folder exists?:", RAW_DIR.exists())
print()
print("Files in raw folder:")
for p in sorted(RAW_DIR.glob("*.json")):
    print("  -", p.name)


## 1. Load Snapshots

Among the files with a collection date, automatically pick the **most recent one**.
(Filenames follow `listings_forsale_YYYY-MM-DD.json`, so sorting puts the latest last.)


In [ ]:
def load_latest(label: str) -> dict:
    """Load the most recent snapshot for the given label ('forsale' or 'sold')."""
    files = sorted(RAW_DIR.glob(f"listings_{label}_*.json"))
    if not files:
        raise FileNotFoundError(f"No '{label}' snapshot found. Run 00_ingest first.")
    latest = files[-1]
    with open(latest, encoding="utf-8") as f:
        snap = json.load(f)
    print(f"[{label}] file: {latest.name}")
    print(f"[{label}] collected_at (UTC): {snap.get('collected_at_utc')}")
    print(f"[{label}] count: {snap.get('count')}")
    return snap

forsale_snap = load_latest("forsale")
sold_snap = load_latest("sold")

forsale = forsale_snap["listings"]
sold = sold_snap["listings"]

print()
print(f"For Sale: {len(forsale)} listings")
print(f"Sold: {len(sold)} listings")
print(f"Total: {len(forsale) + len(sold)} listings")


## 2. Convert to DataFrame

Turn the data into a pandas table for easier analysis.
Combine both groups, keeping a `source` column to mark where each row came from.
(Nested list/dict fields such as `sites` and `metrics` are left as-is here.)


In [ ]:
df_forsale = pd.DataFrame(forsale)
df_forsale["source"] = "For Sale"

df_sold = pd.DataFrame(sold)
df_sold["source"] = "Sold"

df = pd.concat([df_forsale, df_sold], ignore_index=True)

print("Overall shape:", df.shape)   # (rows = listings, cols = fields)
print("Number of columns (fields):", df.shape[1])


## 3. What Fields Exist (Full Enumeration)

List the names of all fields in the response.
Cross-check by eye whether the fields from the feature design table (doc section 5) are actually present.


In [ ]:
cols = sorted(df.columns.tolist())
print(f"{len(cols)} fields total:\n")
for c in cols:
    print(" ", c)


## 4. Per-Field Missing Rate / Coverage

See **how much of each field is filled in**.
- Higher `non_null_%` means the field is well populated.
- Lower means many missing values -> will need missing-value handling if used as a feature.

Note: Amazon-specific fields (e.g. `amazon_sku_count`) only exist for FBA listings, so high missingness is expected there.


In [ ]:
coverage = pd.DataFrame({
    "non_null_count": df.notna().sum(),
    "non_null_%": (df.notna().mean() * 100).round(1),
    "dtype": df.dtypes.astype(str),
})
coverage = coverage.sort_values("non_null_%", ascending=False)

# Keep this option if you want to see the full table
pd.set_option("display.max_rows", None)
coverage


## 5. Target Check — `annual_listing_multiple`

This is the label (answer) for the project: the **listing multiple**.
- How many listings have a value
- What the distribution looks like (min / max / mean)
- Whether there are anomalies (zeros or abnormally large values)

> Note: earlier validation confirmed the sale multiple (`annual_sale_multiple`) is NOT in the API response.
> That is why the primary target was fixed to the listing multiple.


In [ ]:
TARGET = "annual_listing_multiple"

print("Target field:", TARGET)
print("Exists:", TARGET in df.columns)
print()

# May arrive as a string (e.g. "3.08"), so convert to numeric
target_numeric = pd.to_numeric(df[TARGET], errors="coerce")

print(f"Listings with a value: {target_numeric.notna().sum()} / {len(df)}")
print(f"Listings missing: {target_numeric.isna().sum()}")
print()
print("Distribution summary:")
print(target_numeric.describe())

# Re-confirm the sale-multiple field really is absent
print()
print("annual_sale_multiple exists?:", "annual_sale_multiple" in df.columns)


## 6. Leakage Check — Price Fields

Since multiple ≈ price ÷ net profit, **including price fields as features leaks the answer.**
Here we only confirm which price fields exist.
**The exclusion rule is enforced in code in the next step (02_clean_features).**


In [ ]:
price_like = [c for c in df.columns if "price" in c.lower()]
print("Price-related fields (leakage risk -> to be excluded from features):")
for c in price_like:
    filled = df[c].notna().mean() * 100
    print(f"  - {c}  (filled {filled:.1f}%)")


## 7. Quick Check of Key Feature Fields

Confirm the core fields from the feature design table are usable, along with their missing rates.


In [ ]:
key_features = [
    # Financial scale
    "average_annual_net_profit", "average_annual_gross_revenue", "average_annual_expenses",
    # Operations
    "hours_worked_per_week", "days_on_marketplace", "first_made_money_at",
    # Categorical
    "monetizations", "niches", "country", "countries",
    # Boolean (trust / risk)
    "has_trademark", "uses_pbn", "private_lender_approved",
    "patent_pending", "patented_design", "patented_utility",
    # Amazon-specific
    "amazon_sku_count", "amazon_parent_asin_count",
    # Text
    "opportunities", "risks", "summary",
    # For derived features
    "profit_margin",
]

rows = []
for c in key_features:
    if c in df.columns:
        rows.append({"feature": c, "exists": True, "non_null_%": round(df[c].notna().mean()*100, 1)})
    else:
        rows.append({"feature": c, "exists": False, "non_null_%": None})

pd.DataFrame(rows)


## 8. Summary

- Target (`annual_listing_multiple`) coverage confirmed
- Sale multiple absent from the response -> listing multiple fixed as the primary target
- Price fields identified -> to be enforced as an exclusion rule in the next step
- Existence / missing rate of key feature fields confirmed

**Next: `02_clean_features`** — type casting, derived features (business age, margin, etc.),
codifying the price-field exclusion rule, and log-transforming the target.
